# Nvidia Nemotron Reasoning Challenge
## Score: .65

In [ ]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))


offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")

sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Kaggle, set Accelerator to GPU before running training.")
print(f"torch={torch.__version__} | cuda={torch.cuda.is_available()} | gpu={torch.cuda.get_device_name(0)}")


In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
SAMPLE_SIZE = 600
SAMPLE_SEED = 42
LORA_RANK = 32
LORA_ALPHA = 32
MAX_SEQ_LEN = 2560
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 6
LR = 1e-4
LORA_DROPOUT = 0.05
RUN_NAME = f"run_push_e{NUM_EPOCHS}_mx{MAX_SEQ_LEN}_s{SAMPLE_SIZE}"
OUTPUT_DIR = f"/kaggle/working/adapter_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default",
]
MODEL_PATH = None
for _mp in _MODEL_CANDIDATES:
    if os.path.isdir(_mp):
        MODEL_PATH = _mp
        print(f"Using local model path: {MODEL_PATH}")
        break
if MODEL_PATH is None:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Using kagglehub model path: {MODEL_PATH}")
DATA_PATH = "/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv"

train_df = pl.read_csv(DATA_PATH)
train_df = train_df.with_columns(
    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("generated_cot").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
)
train_df = train_df.filter(
    pl.col("prompt").str.len_chars() > 0,
    pl.col("answer").str.len_chars() > 0,
    pl.col("generated_cot").str.len_chars() > 0,
)

n_pool = len(train_df)
take = min(SAMPLE_SIZE, n_pool)
if take < n_pool:
    train_df = train_df.sample(n=take, seed=SAMPLE_SEED)

print(f"Training subset: {len(train_df)} rows (pool {n_pool}, cap {SAMPLE_SIZE})")

hf_dataset = Dataset.from_pandas(train_df.to_pandas())
print(f"Run: {RUN_NAME} | rows: {len(hf_dataset)} | steps/epoch≈{len(hf_dataset)//GRAD_ACCUM} | MAX_SEQ_LEN={MAX_SEQ_LEN} | output: {OUTPUT_DIR}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    prompt = example["prompt"]
    answer = example["answer"]
    cot = example["generated_cot"]

    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}"

    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    seed=42,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")


REQUIRED_ZIP = {"adapter_config.json", "adapter_model.safetensors"}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED_ZIP:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.isfile(fpath):
            raise FileNotFoundError(f"Missing {fpath}")
        zf.write(fpath, arcname=fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("submission.zip is ready")